# Deep Challenge Math Train 필터링 검증

## tl;dr

- 원본 17,000개 샘플을 행 단위로 판정하고 고위험 후보를 원문 검토했다.
- 명백한 결함이 있는 472개를 제거하고 16,528개를 보존했다.
- 보존 우선 원칙을 사용했으므로, 남아 있는 모든 수학 라벨의 정답성을 증명하는 전수 해설 검증은 아니다.


## Context & Methods

목적은 텍스트만으로 풀 수 없는 문항, 불필요하거나 소실된 문구가 섞인 문항, 단일 정수 라벨과 구조적으로 맞지 않는 문항, 그리고 독립 계산으로 명백히 확인된 라벨 불일치를 제외하는 것이다.

판정은 필수 필드·ID·정답 형식 검증, 제어문자/마크업/외부 시각자료 신호 탐지, 번역·운영 지시문 탐지, 다중 출력 구조 탐지, 수동 검토 목록, 독립 계산한 라벨 불일치 목록을 결합했다. 불확실한 행은 삭제하지 않았다.


In [1]:
from pathlib import Path
import csv
import hashlib
import json
from collections import Counter

import pandas as pd

ROOT = Path.cwd()
RAW = ROOT / 'data' / 'deep_chal_math_train.csv'
FILTERED = ROOT / 'data' / 'deep_chal_math_train_filtered.csv'
AUDIT = ROOT / 'data' / 'deep_chal_math_train_filter_audit.csv'
SUMMARY = ROOT / 'report' / 'filtering' / 'filter_summary.json'

def read_csv(path):
    with path.open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

raw = read_csv(RAW)
filtered = read_csv(FILTERED)
audit = read_csv(AUDIT)
summary = json.loads(SUMMARY.read_text(encoding='utf-8'))

print('source_sha256:', hashlib.sha256(RAW.read_bytes()).hexdigest())
print('rows:', {'raw': len(raw), 'filtered': len(filtered), 'audit': len(audit)})


source_sha256: e240dcd9752d12143162706cee4818d4025456605c991ece337df6e9abeb869a
rows: {'raw': 17000, 'filtered': 16528, 'audit': 17000}


## Data

In [2]:
profile = {
    'columns': list(raw[0]),
    'unique_ids': len({r['id'] for r in raw}),
    'blank_id': sum(not r['id'].strip() for r in raw),
    'blank_question': sum(not r['question'].strip() for r in raw),
    'blank_answer': sum(not r['answer'].strip() for r in raw),
    'exact_duplicate_questions': len(raw) - len({r['question'] for r in raw}),
    'integer_answers': sum(r['answer'].lstrip('-').isdigit() for r in raw),
}
pd.DataFrame([profile]).T.rename(columns={0: 'value'})


,value
columns,"[id, question, answer]"
unique_ids,17000
blank_id,0
blank_question,0
blank_answer,0
exact_duplicate_questions,0
integer_answers,17000


## Results

In [3]:
raw_ids = [r['id'] for r in raw]
filtered_ids = [r['id'] for r in filtered]
audit_by_id = {r['id']: r for r in audit}
removed_ids = {r['id'] for r in audit if r['decision'] == 'remove'}

assert len(raw) == 17_000
assert len(audit) == len(raw)
assert len(filtered) + len(removed_ids) == len(raw)
assert len(raw_ids) == len(set(raw_ids))
assert not (set(filtered_ids) & removed_ids)
assert filtered_ids == [row_id for row_id in raw_ids if row_id not in removed_ids]
assert hashlib.sha256(RAW.read_bytes()).hexdigest() == summary['source']['sha256']
assert len(filtered) == summary['decision_counts']['keep']
assert len(removed_ids) == summary['decision_counts']['remove']

reason_counts = Counter(
    r['primary_reason'] for r in audit if r['decision'] == 'remove'
)
reason_table = pd.DataFrame(
    [
        {
            'reason': reason,
            'removed_rows': count,
            'share_of_removed_pct': round(100 * count / len(removed_ids), 2),
            'share_of_source_pct': round(100 * count / len(raw), 3),
        }
        for reason, count in reason_counts.most_common()
    ]
)
print('All integrity assertions passed.')
reason_table


All integrity assertions passed.


,reason,removed_rows,share_of_removed_pct,share_of_source_pct
0,external_visual_dependency,181,38.35,1.065
1,multiple_output_prompt,170,36.02,1.000
2,translation_or_admin_instruction_noise,46,9.75,0.271
3,incomplete_or_corrupt_prompt,24,5.08,0.141
4,verified_label_mismatch,17,3.60,0.100
5,answer_leakage,17,3.60,0.100
6,control_character_corruption,15,3.18,0.088
7,unclosed_visual_markup,2,0.42,0.012


In [4]:
label_mismatches = pd.DataFrame(
    [
        {
            'id': r['id'],
            'source_label': r['answer'],
            'evidence': r['evidence'],
        }
        for r in audit
        if 'verified_label_mismatch' in r['reason_codes']
    ]
)
label_mismatches


,id,source_label,evidence
0,train-000284,47190,label=47190; independently_checked=43758; C(18...
1,train-001261,1,label=1; independently_checked=3/4; (21/28)*(1...
2,train-002020,1,label=1; independently_checked=1/8281; the pro...
3,train-002691,1006,label=1006; independently_checked=1015; the le...
4,train-003196,1952,label=1952; independently_checked=48; (28+x/69...
5,train-004448,408408,label=408408; independently_checked=19448; 17!...
6,train-004479,146,label=146; independently_checked=138; 138 is t...
7,train-005616,258048,label=258048; independently_checked=0; no inte...
8,train-005884,1,label=1; independently_checked=not 1; (-3/5+4i...
9,train-011122,5865864355,label=5865864355; independently_checked=586586...


## Takeaways

가장 큰 제거 사유는 외부 시각자료 의존과 단일 정수 라벨에 맞지 않는 다중 출력 문항이다. 행별 감사 로그에는 보존/제거 결정, 모든 사유 코드, 근거, 신뢰도, 질문 해시를 남겨 후속 검토와 되돌리기가 가능하다.

### Limitations

- 모든 행은 판정 파이프라인을 통과했고 고위험 후보는 원문 검토했지만, 16,528개 보존 문항 각각에 대해 완전한 수학 풀이를 새로 작성한 것은 아니다.
- 라벨 불일치 제거는 명백한 사례만 포함하는 고정밀 목록이며, 보존 라벨 전체의 무오류를 보장하지 않는다.
- 불확실한 문항은 과삭제를 피하기 위해 보존했다.
